# Feature Engineering

## Goals

- Clean raw features
- Handle missing values
- Create temporal features
- Create domain-specific features
- Prepare dataset for machine learning

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/raw/public_transport_delays.csv")

df.head()

,trip_id,date,time,transport_type,route_id,origin_station,destination_station,scheduled_departure,scheduled_arrival,actual_departure_delay_min,actual_arrival_delay_min,weather_condition,temperature_C,humidity_percent,wind_speed_kmh,precipitation_mm,event_type,event_attendance_est,traffic_congestion_index,holiday,peak_hour,weekday,season,delayed
0,T00000,2023-01-01,05:00:00,Tram,Route_15,Station_31,Station_6,05:02:00,05:55:00,12,3,Storm,5.1,52,46,13.0,NaN,500,81,0,1,6,Winter,0
1,T00001,2023-01-01,05:15:00,Metro,Route_12,Station_49,Station_32,05:16:00,05:55:00,15,9,Rain,34.0,64,11,11.4,NaN,0,53,0,0,6,Autumn,1
2,T00002,2023-01-01,05:30:00,Bus,Route_16,Station_29,Station_42,05:33:00,06:17:00,0,0,Clear,29.5,35,31,14.1,Sports,0,67,1,0,6,Autumn,0
3,T00003,2023-01-01,05:45:00,Tram,Route_19,Station_26,Station_18,05:49:00,06:08:00,15,10,Clear,27.4,55,41,6.4,NaN,500,84,0,0,6,Winter,1
4,T00004,2023-01-01,06:00:00,Tram,Route_8,Station_18,Station_15,06:00:00,06:35:00,-1,14,Snow,0.1,90,30,18.5,NaN,500,46,0,0,6,Spring,1


In [3]:
df["event_type"] = df["event_type"].fillna("No_Event")

df["event_type"].isnull().sum()

np.int64(0)

In [4]:
df["date"] = pd.to_datetime(df["date"])

df["date"].head()

0   2023-01-01
1   2023-01-01
2   2023-01-01
3   2023-01-01
4   2023-01-01
Name: date, dtype: datetime64[us]

In [5]:
df["date"] = pd.to_datetime(df["date"])

df["date"].head()

0   2023-01-01
1   2023-01-01
2   2023-01-01
3   2023-01-01
4   2023-01-01
Name: date, dtype: datetime64[us]

In [6]:
df["day"] = df["date"].dt.day

df["month"] = df["date"].dt.month

df["day_of_week"] = df["date"].dt.dayofweek

df["is_weekend"] = (
    df["day_of_week"].isin([5, 6])
).astype(int)

df[[
    "date",
    "day",
    "month",
    "day_of_week",
    "is_weekend"
]].head()

,date,day,month,day_of_week,is_weekend
0,2023-01-01,1,1,6,1
1,2023-01-01,1,1,6,1
2,2023-01-01,1,1,6,1
3,2023-01-01,1,1,6,1
4,2023-01-01,1,1,6,1


In [7]:
df["time"] = pd.to_datetime(
    df["time"],
    format="%H:%M:%S"
)

df["hour"] = df["time"].dt.hour

df["minute"] = df["time"].dt.minute

df[[
    "time",
    "hour",
    "minute"
]].head()

,time,hour,minute
0,1900-01-01 05:00:00,5,0
1,1900-01-01 05:15:00,5,15
2,1900-01-01 05:30:00,5,30
3,1900-01-01 05:45:00,5,45
4,1900-01-01 06:00:00,6,0


In [8]:
df["scheduled_departure"] = pd.to_datetime(
    df["scheduled_departure"],
    format="%H:%M:%S"
)

df["scheduled_arrival"] = pd.to_datetime(
    df["scheduled_arrival"],
    format="%H:%M:%S"
)

df["scheduled_trip_duration"] = (
    df["scheduled_arrival"] -
    df["scheduled_departure"]
).dt.total_seconds() / 60

df[[
    "scheduled_departure",
    "scheduled_arrival",
    "scheduled_trip_duration"
]].head()

,scheduled_departure,scheduled_arrival,scheduled_trip_duration
0,1900-01-01 05:02:00,1900-01-01 05:55:00,53.0
1,1900-01-01 05:16:00,1900-01-01 05:55:00,39.0
2,1900-01-01 05:33:00,1900-01-01 06:17:00,44.0
3,1900-01-01 05:49:00,1900-01-01 06:08:00,19.0
4,1900-01-01 06:00:00,1900-01-01 06:35:00,35.0


In [9]:
df["departure_hour"] = df["scheduled_departure"].dt.hour

df["departure_minute"] = df["scheduled_departure"].dt.minute

df["arrival_hour"] = df["scheduled_arrival"].dt.hour

df[[
    "departure_hour",
    "departure_minute",
    "arrival_hour",
    "scheduled_trip_duration"
]].head()

,departure_hour,departure_minute,arrival_hour,scheduled_trip_duration
0,5,2,5,53.0
1,5,16,5,39.0
2,5,33,6,44.0
3,5,49,6,19.0
4,6,0,6,35.0


In [10]:
df[[
    "day",
    "month",
    "day_of_week",
    "is_weekend"
]].nunique()

day            22
month           1
day_of_week     7
is_weekend      2
dtype: int64

In [11]:
df["event_attendance_est"].value_counts().sort_index()

event_attendance_est
0        1216
500       181
2000      210
10000     183
50000     210
Name: count, dtype: int64

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 34 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   trip_id                     2000 non-null   str           
 1   date                        2000 non-null   datetime64[us]
 2   time                        2000 non-null   datetime64[us]
 3   transport_type              2000 non-null   str           
 4   route_id                    2000 non-null   str           
 5   origin_station              2000 non-null   str           
 6   destination_station         2000 non-null   str           
 7   scheduled_departure         2000 non-null   datetime64[us]
 8   scheduled_arrival           2000 non-null   datetime64[us]
 9   actual_departure_delay_min  2000 non-null   int64         
 10  actual_arrival_delay_min    2000 non-null   int64         
 11  weather_condition           2000 non-null   str           
 12  tem

In [13]:
df[[
    "time",
    "scheduled_departure"
]].head(10)

,time,scheduled_departure
0,1900-01-01 05:00:00,1900-01-01 05:02:00
1,1900-01-01 05:15:00,1900-01-01 05:16:00
2,1900-01-01 05:30:00,1900-01-01 05:33:00
3,1900-01-01 05:45:00,1900-01-01 05:49:00
4,1900-01-01 06:00:00,1900-01-01 06:00:00
5,1900-01-01 06:15:00,1900-01-01 06:15:00
6,1900-01-01 06:30:00,1900-01-01 06:34:00
7,1900-01-01 06:45:00,1900-01-01 06:47:00
8,1900-01-01 07:00:00,1900-01-01 07:01:00
9,1900-01-01 07:15:00,1900-01-01 07:17:00


In [14]:
drop_cols = [
    "trip_id",
    "month",

    "date",
    "time",

    "scheduled_departure",
    "scheduled_arrival",

    "hour",
    "minute",

    "actual_departure_delay_min",
    "delayed"
]

model_df = df.drop(columns=drop_cols)

model_df.shape

(2000, 24)

In [15]:
model_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   transport_type            2000 non-null   str    
 1   route_id                  2000 non-null   str    
 2   origin_station            2000 non-null   str    
 3   destination_station       2000 non-null   str    
 4   actual_arrival_delay_min  2000 non-null   int64  
 5   weather_condition         2000 non-null   str    
 6   temperature_C             2000 non-null   float64
 7   humidity_percent          2000 non-null   int64  
 8   wind_speed_kmh            2000 non-null   int64  
 9   precipitation_mm          2000 non-null   float64
 10  event_type                2000 non-null   str    
 11  event_attendance_est      2000 non-null   int64  
 12  traffic_congestion_index  2000 non-null   int64  
 13  holiday                   2000 non-null   int64  
 14  peak_hour          

# Train Test Split

In [16]:
from sklearn.model_selection import train_test_split

X = model_df.drop(columns=["actual_arrival_delay_min"])

y = model_df["actual_arrival_delay_min"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (1600, 23)
X_test : (400, 23)
y_train: (1600,)
y_test : (400,)


# Preprocessing Pipeline

In [17]:
categorical_features = [
    "transport_type",
    "route_id",
    "origin_station",
    "destination_station",
    "weather_condition",
    "event_type",
    "season"
]

numerical_features = [
    "temperature_C",
    "humidity_percent",
    "wind_speed_kmh",
    "precipitation_mm",
    "event_attendance_est",
    "traffic_congestion_index",
    "holiday",
    "peak_hour",
    "weekday",
    "day",
    "day_of_week",
    "is_weekend",
    "departure_hour",
    "departure_minute",
    "arrival_hour",
    "scheduled_trip_duration"
]

print("Categorical:", len(categorical_features))
print("Numerical:", len(numerical_features))

Categorical: 7
Numerical: 16


In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numerical_features
        )
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

# Baseline Model - Linear Regression

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [20]:
lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

lr_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [25]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)

rmse = root_mean_squared_error(
    y_test,
    y_pred
)

r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.3f}")

MAE  : 7.963
RMSE : 9.554
R²   : -0.083


# Random Forest Regressor

In [26]:
from sklearn.ensemble import RandomForestRegressor

In [27]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [28]:
rf_pred = rf_pipeline.predict(X_test)

rf_pred[:5]

array([16.72 , 15.35 , 13.515, 17.495, 14.095])

In [29]:
rf_mae = mean_absolute_error(
    y_test,
    rf_pred
)

rf_rmse = root_mean_squared_error(
    y_test,
    rf_pred
)

rf_r2 = r2_score(
    y_test,
    rf_pred
)

print(f"MAE  : {rf_mae:.3f}")
print(f"RMSE : {rf_rmse:.3f}")
print(f"R²   : {rf_r2:.3f}")

MAE  : 7.888
RMSE : 9.416
R²   : -0.051


In [30]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest"
    ],
    "MAE": [
        mae,
        rf_mae
    ],
    "RMSE": [
        rmse,
        rf_rmse
    ],
    "R2": [
        r2,
        rf_r2
    ]
})

comparison

,Model,MAE,RMSE,R2
0,Linear Regression,7.962882,9.55449,-0.082708
1,Random Forest,7.888125,9.41552,-0.051441


In [31]:
df["delayed"].value_counts()

delayed
1    1499
0     501
Name: count, dtype: int64